# Managing, Deploying, and Scaling Machine Learning Pipelines with Apache Spark

# Model Management

In [ ]:
# !pip install mlflow

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession
  .builder
  .appName("MLPipelineExampleApp")
  .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 08:35:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

filePath = """../data/learning-spark-v2/sf-airbnb/sf-airbnb-clean.parquet"""
airbnbDF = spark.read.parquet(filePath)
(trainDF, testDF) = airbnbDF.randomSplit([.8, .2], seed=42)

categoricalCols = [field for (field, dataType) in trainDF.dtypes if dataType == "string"]
indexOutputCols = [x + "Index" for x in categoricalCols]
stringIndexer = StringIndexer(inputCols=categoricalCols,
  outputCols=indexOutputCols,
  handleInvalid="skip")
numericCols = [field for (field, dataType) in trainDF.dtypes if ((dataType == "double") & (field != "price"))]
assemblerInputs = indexOutputCols + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs,
  outputCol="features")

rf = RandomForestRegressor(labelCol="price", maxBins=40, maxDepth=5,
  numTrees=100, seed=42)

pipeline = Pipeline(stages=[stringIndexer, vecAssembler, rf])

In [ ]:
import mlflow
import mlflow.spark
import pandas as pd

with mlflow.start_run(run_name="random-forest") as run:
  # Log params: num_trees and max_depth
  mlflow.log_param("num_trees", rf.getNumTrees())
  mlflow.log_param("max_depth", rf.getMaxDepth())
  
  # Log model
  pipelineModel = pipeline.fit(trainDF)
  mlflow.spark.log_model(pipelineModel, "model")

  # Log metrics: RMSE and R2
  predDF = pipelineModel.transform(testDF)
  regressionEvaluator = RegressionEvaluator(predictionCol="prediction", labelCol="price")
  rmse = regressionEvaluator.setMetricName("rmse").evaluate(predDF)
  r2 = regressionEvaluator.setMetricName("r2").evaluate(predDF)
  mlflow.log_metrics({"rmse": rmse, "r2": r2})

  # Log artifact: feature importance scores
  rfModel = pipelineModel.stages[-1]
  pandasDF = (pd.DataFrame(list(zip(vecAssembler.getInputCols(),
    rfModel.featureImportances)),
    columns=["feature", "importance"])
    .sort_values(by="importance", ascending=False))
  # First write to local filesystem, then tell MLflow where to find that file
  pandasDF.to_csv("feature-importance.csv", index=False)
  mlflow.log_artifact("feature-importance.csv")

2026/04/22 07:48:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/22 07:48:05 INFO mlflow.store.db.utils: Updating database tables
2026/04/22 07:48:10 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a rais

In [ ]:
# !mlflow ui

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
runs = client.search_runs(run.info.experiment_id,
  order_by=["attributes.start_time desc"],
  max_results=1)
run_id = runs[0].info.run_id
runs[0].data.metrics

{'rmse': 211.5096898777315, 'r2': 0.22794251914574226}

# Model Deployment Options with MLlib

## Batch

In [ ]:
import mlflow.spark
pipelineModel = mlflow.spark.load_model(f"runs:/{run_id}/model")
# Generate predictions
inputDF = spark.read.parquet("../data/learning-spark-v2/sf-airbnb/sf-airbnb-clean.parquet")
predDF = pipelineModel.transform(inputDF)
predDF.show(5)

2026/04/22 08:03:03 INFO mlflow.spark: URI 'runs:/b48b4ff2b5364aeeaae2f9d6fe5439f1/model/sparkml' does not point to the current DFS.
2026/04/22 08:03:03 INFO mlflow.spark: File 'runs:/b48b4ff2b5364aeeaae2f9d6fe5439f1/model/sparkml' not found on DFS. Will attempt to upload the file.


+-----------------+--------------------+----------------+-------------------------+----------------------+--------+----------+-------------+---------------+------------+---------+--------+----+--------+--------------+-----------------+--------------------+----------------------+-------------------------+---------------------+---------------------------+----------------------+-------------------+-----+-----------+------------+-------+-----------------------+-------------------------+----------------------------+------------------------+------------------------------+-------------------------+----------------------+----------------------+------------------------+---------------------+---------------------------+------------------+--------------+-------------+--------------------+------------------+
|host_is_superhost| cancellation_policy|instant_bookable|host_total_listings_count|neighbourhood_cleansed|latitude| longitude|property_type|      room_type|accommodates|bathrooms|bedrooms|beds

In [ ]:
# Retrieve latest production model
model_name = 'model-1'
model_production_uri = f"models:/{model_name}/production"
model_production = mlflow.spark.load_model(model_production_uri)

2026/04/22 08:20:51 INFO mlflow.spark: URI 'models:/model-1/production/sparkml' does not point to the current DFS.
2026/04/22 08:20:51 INFO mlflow.spark: File 'models:/model-1/production/sparkml' not found on DFS. Will attempt to upload the file.


## Streaming

In [ ]:
pipelineModel = mlflow.spark.load_model(f"runs:/{run_id}/model")
repartitionedPath = "../data/learning-spark-v2/sf-airbnb/sf-airbnb-clean-100p.parquet"
schema = spark.read.parquet(repartitionedPath).schema

streamingData = (spark
  .readStream
  .schema(schema) # Can set the schema this way
  .option("maxFilesPerTrigger", 1)
  .parquet(repartitionedPath))

streamPred = pipelineModel.transform(streamingData)

2026/04/22 08:22:21 INFO mlflow.spark: URI 'runs:/b48b4ff2b5364aeeaae2f9d6fe5439f1/model/sparkml' does not point to the current DFS.
2026/04/22 08:22:21 INFO mlflow.spark: File 'runs:/b48b4ff2b5364aeeaae2f9d6fe5439f1/model/sparkml' not found on DFS. Will attempt to upload the file.


In [ ]:
# streamPred.writeStream.format('console').start()

## Real-Time

In [ ]:
# XGBoost

# Leveraging Spark for Non-MLlib Models

## Pandas UDFs

## Spark for Distributed Hyperparameter Tuning

In [ ]:
# Joblib, Hyperopt

# Cleanup

In [ ]:
spark.stop()

NameError: name 'spark' is not defined